In [53]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf transformers torch

In [54]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [55]:
   !pip install -q streamlit langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

In [56]:
%%writefile app.py
import os
import tempfile
import torch
import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

st.set_page_config(page_title="Lakhas", page_icon="📄", layout="centered")

st.markdown(
    """
<style>
.stApp {background: linear-gradient(180deg, #F4F7FF 0%, #FFFFFF 45%, #EEF3FF 100%);}
.stApp, .stApp p, .stApp li, .stApp label, .stApp span, .stApp h1, .stApp h2, .stApp h3, .stApp h4 {color: #0B1F4B;}
header[data-testid="stHeader"] {background: transparent;}
.hero {
    background: radial-gradient(circle at 20% 15%, #2F5BEA 0%, #0B1F4B 55%, #07122E 100%);
    border-radius: 26px;
    padding: 46px 24px;
    text-align: center;
    box-shadow: 0 18px 50px rgba(37, 78, 200, .38), inset 0 0 0 1px rgba(255, 255, 255, .10);
    margin-bottom: 30px;
}
.hero h1 {color: #FFFFFF !important; font-size: 3.2rem; margin: 0; letter-spacing: 3px; text-shadow: 0 0 24px rgba(120, 160, 255, .65);}
.hero p {color: #C9D8FF !important; margin-top: 10px; font-size: 1.05rem;}
div.stButton {display: flex; justify-content: center; margin-top: 8px;}
div.stButton > button {
    background: linear-gradient(135deg, #0B1F4B, #2447B8);
    border: none;
    border-radius: 14px;
    padding: .7rem 2.6rem;
    font-weight: 700;
    box-shadow: 0 8px 24px rgba(36, 71, 184, .40);
    transition: all .2s ease;
}
div.stButton > button p {color: #FFFFFF !important;}
div.stButton > button:hover {transform: translateY(-2px); box-shadow: 0 12px 34px rgba(36, 71, 184, .60);}
div[data-baseweb="tab-list"] {justify-content: center; gap: 10px;}
button[data-baseweb="tab"] p {font-weight: 600;}
[data-testid="stFileUploaderDropzone"] {background: #FFFFFF; border: 2px dashed #2447B8; border-radius: 18px;}
[data-testid="stVerticalBlockBorderWrapper"] {border-radius: 18px; background: #FFFFFF; box-shadow: 0 10px 30px rgba(11, 31, 75, .10);}
.stTextInput input, .stTextArea textarea {border-radius: 12px;}
</style>
<div class="hero">
    <h1>Lakhas</h1>
    <p>لخّص — Upload a document and get the useful parts in seconds</p>
</div>
""",
    unsafe_allow_html=True,
)


@st.cache_resource(show_spinner="Loading models (first time only)...")
def load_models():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    use_gpu = torch.cuda.is_available()
    pipe = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-3B-Instruct" if use_gpu else "Qwen/Qwen2.5-1.5B-Instruct",
        dtype=torch.float16 if use_gpu else torch.float32,
        device_map="auto" if use_gpu else None,
    )
    pipe.tokenizer.padding_side = "left"
    if pipe.tokenizer.pad_token is None:
        pipe.tokenizer.pad_token = pipe.tokenizer.eos_token
    return embeddings, pipe


embeddings, pipe = load_models()

SYSTEM = (
    "You are an expert document analyst. Use only the information in the document. "
    "Be accurate, concise and well organized."
)


def generate_many(prompts, max_new_tokens=400):
    batch = [
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}]
        for p in prompts
    ]
    outs = pipe(
        batch,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        batch_size=3,
    )
    results = []
    for o in outs:
        item = o[0] if isinstance(o, list) else o
        text = item["generated_text"]
        if isinstance(text, list):
            text = text[-1]["content"]
        results.append(text.strip())
    return results


def generate(prompt, max_new_tokens=800):
    return generate_many([prompt], max_new_tokens)[0]


@st.cache_resource(show_spinner="Reading your file...")
def build_db(file_bytes, file_name):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(file_bytes)
        tmp_path = tmp.name
    pages = PyPDFLoader(tmp_path).load()
    os.remove(tmp_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = [c for c in splitter.split_documents(pages) if c.page_content.strip()]
    if not chunks:
        return None, []
    return FAISS.from_documents(chunks, embeddings), chunks


def sample_text(chunks, max_chunks=14):
    if len(chunks) <= max_chunks:
        picked = chunks
    else:
        step = len(chunks) / max_chunks
        picked = [chunks[int(i * step)] for i in range(max_chunks)]
    return "\n\n".join(c.page_content for c in picked)


def batch_chunks(chunks, size=9):
    return [chunks[i:i + size] for i in range(0, len(chunks), size)]


def condense_prompt(text, focus):
    focus_part = f"The reader especially cares about: {focus.strip()}\n\n" if focus.strip() else ""
    return (
        f"{focus_part}Rewrite the following part of a document as short, dense notes in English. "
        "Keep every important fact: names, dates, amounts, numbers, conditions, obligations, deadlines, decisions and risks. "
        "Drop only filler and repetition. Use short bullet points.\n\n"
        f"{text}"
    )


def summarize(chunks, focus):
    full_text = "\n\n".join(c.page_content for c in chunks)
    material = full_text

    if len(full_text) > 9000:
        parts = batch_chunks(chunks)
        prompts = [condense_prompt("\n\n".join(c.page_content for c in p), focus) for p in parts]
        notes = generate_many(prompts, 350)
        material = "\n\n".join(notes)
        while len(material) > 9000 and len(notes) > 1:
            groups = [notes[i:i + 4] for i in range(0, len(notes), 4)]
            notes = generate_many([condense_prompt("\n\n".join(g), focus) for g in groups], 350)
            material = "\n\n".join(notes)

    structure = (
        "## Overview\n(2-3 sentences on what the document is about)\n\n"
        "## Key Points\n(cover every important point and every section of the document, one short line each)\n\n"
        "## Important Details\n(all dates, amounts, names, deadlines as short bullet points)\n\n"
        "## Risks & Obligations\n(bullet points, or 'None found')\n"
    )
    focus_part = ""
    if focus.strip():
        focus_part = f"Pay special attention to: {focus.strip()}\n\n"
        structure += f"\n## Focus: {focus.strip()}\n(detailed findings about this topic)\n"

    prompt = (
        f"{focus_part}Below is the text of the document (or condensed notes covering all of it). "
        "Write a well-organized report in clear professional English using exactly this Markdown structure. "
        "Do not leave out any important point, but keep every line short.\n\n"
        f"{structure}\nText:\n{material}"
    )
    return generate(prompt, 1000)


def answer_question(db, question):
    docs = db.similarity_search(question, k=6)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = (
        f"Context from the document:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer the question accurately using only the context. Be specific and include exact numbers, names and dates when they appear. "
        "If the context does not contain the answer, say that it was not found in the document. "
        "Answer in the same language as the question."
    )
    return generate(prompt, 400)


def compare(chunks1, chunks2, name1, name2):
    prompt = (
        "Compare the two documents below and write the result in clear English using exactly this Markdown structure:\n\n"
        "## Similarities\n(bullet points)\n\n"
        "## Key Differences\n(bullet points)\n\n"
        "## Conclusion\n(2-3 sentences)\n\n"
        f"Document 1 ({name1}):\n{sample_text(chunks1, 8)}\n\n"
        f"Document 2 ({name2}):\n{sample_text(chunks2, 8)}"
    )
    return generate(prompt, 700)


files = st.file_uploader("Upload one or more PDF files", type="pdf", accept_multiple_files=True)

if not files:
    st.info("Upload at least one PDF to get started")
    st.stop()

dbs = {}
all_chunks = {}
for f in files:
    db, chunks = build_db(f.getvalue(), f.name)
    if db is None:
        st.error(f"Could not read any text from {f.name}. It may be a scanned PDF.")
        continue
    dbs[f.name] = db
    all_chunks[f.name] = chunks

if not dbs:
    st.stop()

names = list(dbs.keys())
selected = st.selectbox("Choose a document", names)

tab_sum, tab_ask, tab_cmp = st.tabs(["✨ Summary", "💬 Ask", "⚖️ Compare"])

with tab_sum:
    focus = st.text_input("What should Lakhas focus on? (optional)", placeholder="e.g. payment terms, deadlines, penalties")
    if st.button("Summarize", type="primary"):
        key = f"{selected}||{focus.strip()}"
        cache = st.session_state.setdefault("summaries", {})
        if key not in cache:
            with st.spinner("Lakhas is reading and summarizing..."):
                cache[key] = summarize(all_chunks[selected], focus)
        with st.container(border=True):
            st.markdown(cache[key])

with tab_ask:
    question = st.text_input("Ask anything about the document", placeholder="e.g. What is the contract duration?")
    if st.button("Get answer"):
        if not question.strip():
            st.warning("Type a question first")
        else:
            with st.spinner("Searching the document..."):
                answer = answer_question(dbs[selected], question)
            with st.container(border=True):
                st.markdown(answer)

with tab_cmp:
    if len(names) < 2:
        st.info("Upload at least two documents to compare them")
    else:
        first = st.selectbox("First document", names, index=0)
        second = st.selectbox("Second document", names, index=1)
        if st.button("Compare"):
            if first == second:
                st.warning("Choose two different documents")
            else:
                with st.spinner("Comparing..."):
                    result = compare(all_chunks[first], all_chunks[second], first, second)
                with st.container(border=True):
                    st.markdown(result)

Overwriting app.py


In [57]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

cloudflared: Text file busy


In [58]:
import subprocess, time, re

subprocess.run(["pkill", "-f", "streamlit"])
subprocess.run(["pkill", "-f", "cloudflared"])
time.sleep(2)

subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--theme.base", "light", "--theme.primaryColor", "#2447B8"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(15)

subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.DEVNULL, stderr=open("cf.log", "w"),
)
time.sleep(10)

print(re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read()).group(0))

https://verse-ballot-shots-reliable.trycloudflare.com
